In [4]:
pip install requests pandas selenium webdriver-manager

  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached 

In [ ]:
import os
import time
import random
import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- [1. 발급받으신 인증 정보 적용] ---
CLIENT_ID = "xaycKafg5L6rPQFiuw0x"
CLIENT_SECRET = "nYRS3j1EhY"

CHUNK_SIZE = 10000  # 1만 건당 파일 분할 저장
LOG_FILE = "collected_books_log.txt"  # 중복 수집 방지용 로그

# --- [2. 브라우저 및 로그 설정] ---
def setup_driver():
    options = Options()
    options.add_argument('--headless')  # 백그라운드 실행
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def load_log():
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r', encoding='utf-8') as f:
            return set(line.strip() for line in f)
    return set()

def write_log(book_link):
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(book_link + "\n")

# --- [3. 리뷰 추출 함수] ---
def get_reviews(driver, title):
    reviews = []
    try:
        # 리뷰 요소가 나타날 때까지 대기
        WebDriverWait(driver, 4).until(
            EC.presence_of_element_located((By.CLASS_NAME, "comment_text_box"))
        )
        elements = driver.find_elements(By.CLASS_NAME, "comment_text_box")
        for el in elements:
            text = el.text.strip()
            if text:
                reviews.append({
                    '도서명': title,
                    '리뷰내용': text,
                    '수집일자': time.strftime('%Y-%m-%d')
                })
    except:
        pass # 리뷰가 없는 도서는 스킵
    return reviews

# --- [4. 메인 실행 루프] ---
def main():
    driver = setup_driver()
    collected_books = load_log()
    all_temp_data = []
    total_count = 0
    file_num = 1
    
    # 최대한 많은 도서를 훑기 위한 광범위한 카테고리 키워드
    categories = ["소설", "경영", "자기계발", "인문", "사회", "과학", "역사", "예술", "종교", "만화"]

    try:
        for cat in categories:
            print(f"\n[카테고리: {cat}] 수집 시작...")
            
            # 네이버 도서 검색 API 호출 (한 번에 100권)
            url = f"https://openapi.naver.com/v1/search/book.json?query={cat}&display=100"
            headers = {"X-Naver-Client-Id": CLIENT_ID, "X-Naver-Client-Secret": CLIENT_SECRET}
            res = requests.get(url, headers=headers).json()
            items = res.get('items', [])

            if not items:
                print(f" > {cat} 결과가 없습니다. API 설정을 확인하세요.")
                continue

            for item in items:
                link = item['link']
                title = item['title'].replace('<b>', '').replace('</b>', '')

                if link in collected_books:
                    continue

                driver.get(link)
                time.sleep(random.uniform(1.0, 2.0)) # 차단 방지용 딜레이
                
                reviews = get_reviews(driver, title)
                if reviews:
                    all_temp_data.extend(reviews)
                    total_count += len(reviews)
                    print(f" > 수집 중: {title[:15]}... (현재 {total_count}건)")

                # 수집 기록 저장 및 로그 업데이트
                collected_books.add(link)
                write_log(link)

                # 1만 건 단위 자동 분할 저장
                if len(all_temp_data) >= CHUNK_SIZE:
                    df = pd.DataFrame(all_temp_data)
                    filename = f"naver_reviews_part_{file_num:03d}.csv"
                    df.to_csv(filename, index=False, encoding='utf-8-sig')
                    print(f"\n✅ {filename} 저장 완료!\n")
                    all_temp_data = [] # 리스트 초기화
                    file_num += 1

    except Exception as e:
        print(f"\n❌ 작업 중단 에러: {e}")
    finally:
        # 종료 전 남은 데이터 최종 저장
        if all_temp_data:
            filename = f"naver_reviews_part_{file_num:03d}_final.csv"
            pd.DataFrame(all_temp_data).to_csv(filename, index=False, encoding='utf-8-sig')
            print(f"\n✅ 최종 파일 저장 완료: {filename}")
        
        driver.quit()
        print(f"\n🏁 모든 작업이 완료되었습니다. 총 {total_count}건 수집.")

if __name__ == "__main__":
    main()


[카테고리: 소설] 수집 시작...

[카테고리: 경영] 수집 시작...

[카테고리: 자기계발] 수집 시작...

[카테고리: 인문] 수집 시작...
